# 第0回: PyTorch の基礎

## カリキュラム一覧

| 回 | テーマ |
|:--:|:-------|
| **0** | **PyTorchの基礎** ← 今回 |
| 1 | EIPL前処理 |
| 2 | PyTorchのLayer理解 (BatchNorm, LayerNorm, SpatialSoftmax, Transformer) |
| 3 | CAE (MNISTによる画像圧縮・再構成) |
| 4 | Robomimicデータセット |
| 5 | RNN (Robomimic学習) |
| 6 | Mamba (Robomimic学習) |
| 7 | CNNRNN (Robomimic学習) |
| 8 | SARNN (Robomimic学習) |
| 9 | HSARNN (Robomimic学習) |
| 10 | StochasticRNN (Robomimic学習) |
| 11 | Diffusion Policy (Robomimic学習) |
| 12 | Flow Matching (Robomimic学習) |
| 13 | ACT (Robomimic学習) |

## はじめに

このノートブックでは、PyTorch の基本的な使い方を学びます。
テンソルの作成・操作から始めて、ニューラルネットワークの構築、自動微分、そして学習ループの実装までをカバーします。

## 1. 環境セットアップ

必要なライブラリをインストールします。Google Colab を使用している場合は以下のセルを実行してください。

In [ ]:
!pip install -q torch torchvision numpy matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
from tqdm.notebook import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. テンソルの基礎

テンソル (Tensor) は PyTorch の基本的なデータ構造です。NumPy の ndarray に似ていますが、GPU 上で計算できる点が大きな特徴です。

### 2.1 テンソルの作成

In [ ]:
# さまざまな方法でテンソルを作成する

# Python リストから作成
x = torch.tensor([1, 2, 3])
print("リストから:", x)

# ゼロで初期化
zeros = torch.zeros(3, 4)
print("\nゼロテンソル:")
print(zeros)

# 1 で初期化
ones = torch.ones(2, 3)
print("\n1テンソル:")
print(ones)

# ランダムな値
rand = torch.randn(2, 3)  # 標準正規分布
print("\nランダムテンソル:")
print(rand)

# 等間隔の値
linspace = torch.linspace(0, 1, steps=5)
print("\n等間隔:", linspace)

# 単位行列
eye = torch.eye(3)
print("\n単位行列:")
print(eye)

### 2.2 テンソルの属性

In [ ]:
x = torch.randn(3, 4, 5)

print(f"形状 (shape): {x.shape}")
print(f"次元数 (ndim): {x.ndim}")
print(f"データ型 (dtype): {x.dtype}")
print(f"デバイス (device): {x.device}")
print(f"要素数 (numel): {x.numel()}")

### 2.3 形状の操作

In [ ]:
x = torch.arange(12)
print("元のテンソル:", x)
print("形状:", x.shape)

# reshape
y = x.reshape(3, 4)
print("\nreshape(3, 4):")
print(y)

# view (メモリが連続している場合のみ使用可能)
z = x.view(4, 3)
print("\nview(4, 3):")
print(z)

# unsqueeze: 次元を追加
a = torch.tensor([1, 2, 3])
print("\n元:", a.shape)
print("unsqueeze(0):", a.unsqueeze(0).shape)  # (1, 3)
print("unsqueeze(1):", a.unsqueeze(1).shape)  # (3, 1)

# squeeze: サイズ1の次元を削除
b = torch.zeros(1, 3, 1)
print("\n元:", b.shape)
print("squeeze():", b.squeeze().shape)

# transpose / permute
c = torch.randn(2, 3, 4)
print("\n元:", c.shape)
print("permute(2,0,1):", c.permute(2, 0, 1).shape)

### 2.4 テンソルの演算

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# 基本演算
print("加算:", a + b)
print("減算:", a - b)
print("乗算:", a * b)
print("除算:", a / b)
print("べき乗:", a ** 2)

# 行列演算
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B  # 行列積
print(f"\n行列積: {A.shape} @ {B.shape} = {C.shape}")

# 集約演算
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"\n合計: {x.sum()}")
print(f"平均: {x.mean()}")
print(f"最大値: {x.max()}")
print(f"軸ごとの合計 (dim=0): {x.sum(dim=0)}")
print(f"軸ごとの合計 (dim=1): {x.sum(dim=1)}")

### 2.5 インデキシングとスライシング

In [ ]:
x = torch.arange(12).reshape(3, 4)
print("元のテンソル:")
print(x)

print(f"\nx[0]: {x[0]}")
print(f"x[1, 2]: {x[1, 2]}")
print(f"x[:, 1]: {x[:, 1]}")
print(f"x[0:2, 1:3]:")
print(x[0:2, 1:3])

# ブーリアンインデキシング
mask = x > 5
print(f"\n5より大きい要素: {x[mask]}")

## 3. NumPy との変換

PyTorch テンソルと NumPy 配列は簡単に相互変換できます。
**注意:** CPU 上のテンソルと NumPy 配列はメモリを共有するため、一方を変更するともう一方にも反映されます。

In [ ]:
# Tensor -> NumPy
t = torch.ones(3)
n = t.numpy()
print(f"Tensor: {t}")
print(f"NumPy:  {n}")

# メモリの共有を確認
t[0] = 42
print(f"\nTensor変更後:")
print(f"Tensor: {t}")
print(f"NumPy:  {n}")  # NumPy も変わる

In [ ]:
# NumPy -> Tensor
n = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(n)
print(f"NumPy:  {n}")
print(f"Tensor: {t}")

# メモリの共有を確認
n[0] = 99
print(f"\nNumPy変更後:")
print(f"NumPy:  {n}")
print(f"Tensor: {t}")  # Tensor も変わる

## 4. デバイス管理 (CPU / GPU)

PyTorch ではテンソルを CPU と GPU の間で自在に移動させることができます。
GPU を使うことで大規模な計算を高速に実行できます。

In [ ]:
# デバイスの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# テンソルをデバイスに移動
x = torch.randn(3, 3)
x_device = x.to(device)
print(f"テンソルのデバイス: {x_device.device}")

# 作成時にデバイスを指定
y = torch.ones(3, 3, device=device)
print(f"作成時に指定: {y.device}")

# GPU テンソルを NumPy に変換するには、まず CPU に戻す
if device.type == "cuda":
    arr = x_device.cpu().numpy()
else:
    arr = x_device.numpy()
print(f"NumPy変換成功: {arr.shape}")

## 5. nn.Module の基礎

`nn.Module` は PyTorch でニューラルネットワークを構築するための基底クラスです。
すべてのモデルは `nn.Module` を継承して作成します。

### 5.1 基本的なネットワーク

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# モデルの作成
model = SimpleNet(input_dim=10, hidden_dim=32, output_dim=3)
print(model)

In [ ]:
# パラメータの確認
print("パラメータ一覧:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\n総パラメータ数: {total_params}")

In [ ]:
# 順伝播 (forward pass)
x = torch.randn(5, 10)  # バッチサイズ5, 入力次元10
output = model(x)
print(f"入力: {x.shape}")
print(f"出力: {output.shape}")

### 5.2 nn.Sequential を使った定義

In [ ]:
# nn.Sequential でシンプルに定義
model_seq = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)
print(model_seq)

output = model_seq(torch.randn(5, 10))
print(f"\n出力形状: {output.shape}")

## 6. 自動微分 (Autograd)

PyTorch の自動微分エンジンは、テンソルに対する演算の勾配を自動的に計算します。
`requires_grad=True` を設定したテンソルに対する全ての演算が追跡されます。

In [ ]:
# requires_grad=True で勾配追跡を有効化
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x = {x}")

# 演算を行う
y = x ** 2 + 3 * x + 1
z = y.sum()
print(f"y = x^2 + 3x + 1 = {y}")
print(f"z = sum(y) = {z}")

# 逆伝播 (backward pass)
z.backward()

# 勾配を確認: dz/dx = 2x + 3
print(f"\n勾配 dz/dx = 2x + 3 = {x.grad}")
print(f"検算: 2*{x.data} + 3 = {2 * x.data + 3}")

In [ ]:
# 勾配の蓄積に注意
x = torch.tensor([1.0], requires_grad=True)

# 1回目
y1 = (x * 2).sum()
y1.backward()
print(f"1回目の勾配: {x.grad}")

# 2回目 (勾配が蓄積される！)
y2 = (x * 3).sum()
y2.backward()
print(f"2回目の勾配 (蓄積): {x.grad}")  # 2 + 3 = 5

# 勾配をリセット
x.grad.zero_()
y3 = (x * 3).sum()
y3.backward()
print(f"リセット後の勾配: {x.grad}")  # 3

In [ ]:
# 勾配追跡の無効化 (推論時に使用)
x = torch.randn(3, requires_grad=True)

# torch.no_grad() コンテキスト
with torch.no_grad():
    y = x * 2
    print(f"no_grad 内: requires_grad = {y.requires_grad}")

# .detach() メソッド
z = x.detach()
print(f"detach 後: requires_grad = {z.requires_grad}")

## 7. 損失関数とオプティマイザ

### 7.1 損失関数

損失関数はモデルの出力と正解ラベルとの差を計測します。

In [ ]:
# よく使われる損失関数

# MSE Loss (回帰タスク)
mse_loss = nn.MSELoss()
pred = torch.randn(5, 1)
target = torch.randn(5, 1)
loss_mse = mse_loss(pred, target)
print(f"MSE Loss: {loss_mse.item():.4f}")

# Cross Entropy Loss (分類タスク)
ce_loss = nn.CrossEntropyLoss()
pred_class = torch.randn(5, 3)   # 5サンプル, 3クラス
target_class = torch.tensor([0, 2, 1, 0, 2])  # 正解ラベル
loss_ce = ce_loss(pred_class, target_class)
print(f"Cross Entropy Loss: {loss_ce.item():.4f}")

# Binary Cross Entropy Loss (2クラス分類)
bce_loss = nn.BCEWithLogitsLoss()
pred_bin = torch.randn(5)
target_bin = torch.tensor([1.0, 0.0, 1.0, 1.0, 0.0])
loss_bce = bce_loss(pred_bin, target_bin)
print(f"BCE Loss: {loss_bce.item():.4f}")

### 7.2 オプティマイザ

In [ ]:
model = SimpleNet(10, 32, 3)

# SGD
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam (最もよく使われる)
optimizer_adam = optim.Adam(model.parameters(), lr=0.001)

# AdamW (重み減衰付き Adam)
optimizer_adamw = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

print("オプティマイザの例:")
print(f"  SGD:   lr=0.01, momentum=0.9")
print(f"  Adam:  lr=0.001")
print(f"  AdamW: lr=0.001, weight_decay=0.01")

## 8. 学習ループの実装

ここでは、簡単な回帰問題を例に学習ループを実装します。
$y = 2x + 1$ のデータを生成し、モデルに学習させます。

In [ ]:
# データの生成
torch.manual_seed(42)
n_samples = 200

X = torch.randn(n_samples, 1) * 5
y_true = 2 * X + 1 + torch.randn(n_samples, 1) * 0.5  # ノイズ付き

# データの可視化
plt.figure(figsize=(8, 5))
plt.scatter(X.numpy(), y_true.numpy(), alpha=0.5, s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.title("生成データ: y = 2x + 1 + ノイズ")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# モデルの定義
model = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 学習ループ
n_epochs = 200
losses = []

for epoch in tqdm(range(n_epochs), desc="学習中"):
    # 順伝播
    y_pred = model(X)
    loss = criterion(y_pred, y_true)

    # 逆伝播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

In [ ]:
# 学習曲線の可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 損失の推移
axes[0].plot(losses)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("学習曲線")
axes[0].grid(True, alpha=0.3)

# 予測結果
with torch.no_grad():
    X_sorted, indices = X.sort(dim=0)
    y_pred_sorted = model(X_sorted)

axes[1].scatter(X.numpy(), y_true.numpy(), alpha=0.4, s=10, label="データ")
axes[1].plot(X_sorted.numpy(), y_pred_sorted.numpy(), color="red", linewidth=2, label="モデル予測")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("予測結果")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 演習問題

### 演習1: テンソルの作成と操作

以下のタスクを行ってください:
1. 3×3 の単位行列を作成
2. 0 から 8 までの整数テンソルを作成し、3×3 に reshape
3. 上記2つの行列の行列積を計算
4. 結果の各行の合計を求める

In [ ]:
# 1. 単位行列の作成

# 2. 0~8 の整数テンソルを 3x3 に reshape

# 3. 行列積の計算

# 4. 各行の合計

<details>
<summary><b>回答例を表示</b></summary>

```python
# 1. 単位行列の作成
I = torch.eye(3)
print("単位行列:\n", I)

# 2. 0~8 の整数テンソルを 3x3 に reshape
A = torch.arange(9).reshape(3, 3).float()
print("\n行列A:\n", A)

# 3. 行列積の計算
C = I @ A
print("\n行列積 I @ A:\n", C)

# 4. 各行の合計
row_sum = C.sum(dim=1)
print("\n各行の合計:", row_sum)
```

</details>

### 演習2: カスタム nn.Module の作成

以下の仕様のニューラルネットワークを `nn.Module` を継承して作成してください:
- 入力次元: 5
- 隠れ層1: 32ユニット + ReLU
- 隠れ層2: 16ユニット + ReLU
- 出力次元: 2
- `forward` メソッドを実装
- ランダムな入力 (バッチサイズ=3) を作成して順伝播を確認

In [ ]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにレイヤーを定義

    def forward(self, x):
        # ここに順伝播を実装
        pass

# モデルの作成とテスト

<details>
<summary><b>回答例を表示</b></summary>

```python
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = MyNet()
print(model)
print(f"\n総パラメータ数: {sum(p.numel() for p in model.parameters())}")

x = torch.randn(3, 5)
output = model(x)
print(f"\n入力形状: {x.shape}")
print(f"出力形状: {output.shape}")
print(f"出力:\n{output}")
```

</details>

### 演習3: 自動微分

以下のタスクを行ってください:
1. `requires_grad=True` で $x = 3.0$ のテンソルを作成
2. $f(x) = x^3 + 2x^2 - 5x + 4$ を計算
3. `backward()` で勾配を計算
4. 手計算の結果 $f'(x) = 3x^2 + 4x - 5$ と比較

In [ ]:
# 1. テンソルの作成

# 2. f(x) の計算

# 3. 勾配の計算

# 4. 手計算との比較

<details>
<summary><b>回答例を表示</b></summary>

```python
# 1. テンソルの作成
x = torch.tensor(3.0, requires_grad=True)
print(f"x = {x}")

# 2. f(x) の計算
f = x**3 + 2*x**2 - 5*x + 4
print(f"f(x) = x^3 + 2x^2 - 5x + 4 = {f.item()}")

# 3. 勾配の計算
f.backward()
print(f"\n自動微分による勾配: df/dx = {x.grad.item()}")

# 4. 手計算との比較
# f'(x) = 3x^2 + 4x - 5
manual_grad = 3 * 3.0**2 + 4 * 3.0 - 5
print(f"手計算による勾配: f'(3) = 3*9 + 12 - 5 = {manual_grad}")
print(f"一致: {abs(x.grad.item() - manual_grad) < 1e-6}")
```

</details>

---
## まとめ

このノートブックでは、以下の PyTorch の基礎を学びました:

1. **テンソル**: 作成、属性、形状操作、演算、インデキシング
2. **NumPy との変換**: `tensor.numpy()` と `torch.from_numpy()`（メモリ共有に注意）
3. **デバイス管理**: CPU / GPU 間のテンソル移動
4. **nn.Module**: ニューラルネットワークの構築方法
5. **自動微分 (Autograd)**: `requires_grad`、`backward()`、勾配計算
6. **損失関数とオプティマイザ**: MSE、CrossEntropy、Adam など
7. **学習ループ**: 順伝播 → 損失計算 → 逆伝播 → パラメータ更新

次回の「第1回: EIPL前処理」では、実際のロボティクスデータの前処理について学びます。